## FASE 4 — Análisis Inferencial (A/B Testing)

**Objetivo:** <br> confirmar estadísticamente los hallazgos detectados 
visualmente en el EDA (Fase 3). Cada caso de esta fase parte de una 
pregunta ya planteada en Fase 3, para dar respaldo estadístico formal 
a lo observado.

**Flujo:** <br>normalidad → homocedasticidad → nº de grupos → test 
correspondiente (t-test / Mann-Whitney / ANOVA+Tukey / Kruskal-Wallis).

In [1]:
import sys
sys.path.append('../')
from src import sp_abtest as sa
import pandas as pd
import scipy.stats as stats

df = pd.read_csv("../data/processed/02_datos_limpios.csv", parse_dates=['fecha_pedido'])
df_pedidos = df

### Tablas necesarias (clientes y productos)

In [2]:
clientes = df.groupby('id_cliente').agg(
    gasto_total=('importe_total_wz', 'sum'),
    n_pedidos=('order_id', 'count'),
    ciudad=('ciudad', 'first'),
    segmento_cliente=('segmento_cliente', 'first'),
    ingresos_anuales=('ingresos_anuales', 'first'),
).reset_index()

productos = df.groupby('product_id').agg(
    precio_medio=('precio_unitario', 'mean'),
    costo_medio=('costo_unitario', 'mean'),
    categoria=('categoria_producto', 'first'),
).reset_index()
productos['margen'] = productos['precio_medio'] - productos['costo_medio']

### Caso A — gasto_total ~ segmento_cliente (Cliente, 4 grupos)

**Origen en Fase 3:** en el EDA de Clientes, el gasto medio por 
segmento apenas variaba (671€-689€). Se confirma aquí con test formal.

In [3]:
sa.normalidad(clientes, ['gasto_total'])
sa.homocedasticidad(clientes, 'segmento_cliente', ['gasto_total'])


Para la columna GASTO_TOTAL los datos NO siguen una distribución normal
Para la columna GASTO_TOTAL las varianzas SÍ son homgéneas entre grupos, SI hay HOMOCEDASTICIDAD


In [4]:
sa.decidir_test(clientes, 'segmento_cliente', ['gasto_total'])

--- GASTO_TOTAL | normalidad=False | homocedastico=True | n_grupos=4 ---
H=2.3883, p=0.4958
Para la métrica GASTO_TOTAL, las medianas SI son iguales entre los grupos de segmento_cliente (Kruskal-Wallis no significativo)


In [5]:
sa.kruskal(clientes, 'segmento_cliente', ['gasto_total'])

H=2.3883, p=0.4958
Para la métrica GASTO_TOTAL, las medianas SI son iguales entre los grupos de segmento_cliente (Kruskal-Wallis no significativo)


**Resultado:** Shapiro NO normal, Levene SÍ homocedástico → Kruskal-Wallis. 
H=2,3883; p=0,4958 → **no se rechaza H0**. Confirma lo visto en el EDA: 
segmento_cliente no diferencia el gasto real (medianas: 621-637€).

### Caso B — importe_total_wz ~ canal (Pedido, 4 grupos)

**Origen en Fase 3:** en el EDA de Canales, la mediana de importe era 
casi idéntica entre los 4 canales (71-74€).

In [6]:
sa.normalidad(df_pedidos, ['importe_total_wz'])
sa.homocedasticidad(df_pedidos, 'canal', ['importe_total_wz'])
sa.kruskal(df_pedidos, 'canal', ['importe_total_wz'])

Para la columna IMPORTE_TOTAL_WZ los datos NO siguen una distribución normal
Para la columna IMPORTE_TOTAL_WZ las varianzas SÍ son homgéneas entre grupos, SI hay HOMOCEDASTICIDAD
H=2.9307, p=0.4024
Para la métrica IMPORTE_TOTAL_WZ, las medianas SI son iguales entre los grupos de canal (Kruskal-Wallis no significativo)


In [7]:
sa.decidir_test(df_pedidos, 'canal', ['importe_total_wz'])

--- IMPORTE_TOTAL_WZ | normalidad=False | homocedastico=True | n_grupos=4 ---
H=2.9307, p=0.4024
Para la métrica IMPORTE_TOTAL_WZ, las medianas SI son iguales entre los grupos de canal (Kruskal-Wallis no significativo)


**Resultado:** NO normal → Kruskal-Wallis. H=2,9307; p=0,4024 → 
**no se rechaza H0**. Confirma que el canal no influye en el importe.

### Caso C — importe_total_wz ~ envio_gratis (Pedido, 2 grupos)

**Origen en Fase 3:** contrario a lo esperado, el EDA mostró que 
envío gratis no se asociaba a mayor importe (72,49€ vs 72,60€).

In [8]:
sa.normalidad(df_pedidos, ['importe_total_wz'])
sa.homocedasticidad(df_pedidos, 'envio_gratis', ['importe_total_wz'])
sa.mannwhitneyu(df_pedidos, 'envio_gratis', ['importe_total_wz'])

Para la columna IMPORTE_TOTAL_WZ los datos NO siguen una distribución normal


Para la columna IMPORTE_TOTAL_WZ las varianzas SÍ son homgéneas entre grupos, SI hay HOMOCEDASTICIDAD
U=284698618.0000, p=0.9573
Para la métrica IMPORTE_TOTAL_WZ, las medianas SI son iguales, es decir, NO hay deferencias significativas entre grupos


In [9]:
sa.decidir_test(df_pedidos,'envio_gratis', ['importe_total_wz'])

--- IMPORTE_TOTAL_WZ | normalidad=False | homocedastico=True | n_grupos=2 ---
U=284698618.0000, p=0.9573
Para la métrica IMPORTE_TOTAL_WZ, las medianas SI son iguales, es decir, NO hay deferencias significativas entre grupos


**Resultado:** NO normal → Mann-Whitney. U=284.698.618; p=0,9573 → 
**no se rechaza H0**. Confirma con rigor lo que ya sorprendía en el EDA.

### Caso D — margen ~ categoria_producto

**Origen en Fase 3:** en el EDA de Productos, el margen medio SÍ 
variaba claramente entre categorías (electrónica 14,72€ vs hogar 
29,29€). Este caso se ejecuta a **dos niveles** para mostrar por qué 
la unidad de análisis importa tanto como el test elegido.

#### D.1 — A nivel PRODUCTO (n=90) — NO HACER ASÍ

In [10]:
sa.normalidad(productos, ['margen'])
sa.homocedasticidad(productos, 'categoria', ['margen'])
sa.kruskal(productos, 'categoria', ['margen'])

Para la columna MARGEN los datos NO siguen una distribución normal
Para la columna MARGEN las varianzas SÍ son homgéneas entre grupos, SI hay HOMOCEDASTICIDAD
H=7.4365, p=0.1145
Para la métrica MARGEN, las medianas SI son iguales entre los grupos de categoria (Kruskal-Wallis no significativo)


In [11]:
sa.decidir_test(productos, 'categoria', ['margen'])

--- MARGEN | normalidad=False | homocedastico=True | n_grupos=5 ---
H=7.4365, p=0.1145
Para la métrica MARGEN, las medianas SI son iguales entre los grupos de categoria (Kruskal-Wallis no significativo)


**Resultado D.1:** H=7,4365; p=0,1145 → **NO significativo**. 

⚠️ **Por qué NO se debe concluir aquí:** `margen` es un atributo fijo 
por producto (costo_unitario no varía entre pedidos de un mismo 
producto), así que al construir `productos` (1 fila = 1 producto) solo 
hay **n=90**, repartidos en 5 categorías (~18 productos cada una). Con 
tan poca muestra, aunque la diferencia de medias sea grande en términos 
absolutos, el test no tiene potencia suficiente para declararla 
significativa. Concluir "no hay diferencia" aquí sería un error de 
interpretación, no un hallazgo real.



#### D.2 — A nivel PEDIDO (n=52.000) — versión correcta

In [12]:
df_pedidos['margen_pedido'] = df_pedidos['precio_unitario'] - df_pedidos['costo_unitario']

sa.normalidad(df_pedidos, ['margen_pedido'])
sa.homocedasticidad(df_pedidos, 'categoria_producto', ['margen_pedido'])
sa.kruskal(df_pedidos, 'categoria_producto', ['margen_pedido'])

Para la columna MARGEN_PEDIDO los datos NO siguen una distribución normal
Para la columna MARGEN_PEDIDO las varianzas SÍ son homgéneas entre grupos, SI hay HOMOCEDASTICIDAD
H=1034.1079, p=0.0000
Para la métrica MARGEN_PEDIDO, al menos un grupo de categoria_producto tiene una mediana distinta (Kruskal-Wallis significativo)


In [14]:
sa.decidir_test(df_pedidos, 'categoria_producto', ['margen_pedido'])

--- MARGEN_PEDIDO | normalidad=False | homocedastico=True | n_grupos=5 ---
H=1034.1079, p=0.0000
Para la métrica MARGEN_PEDIDO, al menos un grupo de categoria_producto tiene una mediana distinta (Kruskal-Wallis significativo)


**Resultado D.2:** H=1034,11; p=1,45e-222 → **SIGNIFICATIVO**. 

Repitiendo el mismo margen (constante por producto) a nivel pedido, la 
muestra sube a n=52.000 y el test detecta la diferencia con muchísima 
contundencia. **Es el mismo patrón real en ambos casos — cambia solo 
la potencia estadística disponible para detectarlo.**

**Conclusión del Caso D:** cuando la unidad de análisis natural de una 
variable (producto) tiene poca muestra, conviene replicarla a un nivel 
con más filas (pedido) si esto no distorsiona la pregunta de negocio. 
Aquí es válido porque seguimos preguntando "¿varía el margen entre 
categorías?", no algo distinto por cambiar de nivel.

### Caso E — gasto_total ~ ciudad (Cliente, 10 grupos)

**Origen en Fase 3:** el EDA mostró que el gasto medio por ciudad era 
prácticamente constante (661€-697€).

In [15]:
sa.normalidad(clientes, ['gasto_total'])
sa.homocedasticidad(clientes, 'ciudad', ['gasto_total'])
sa.kruskal(clientes, 'ciudad', ['gasto_total'])

Para la columna GASTO_TOTAL los datos NO siguen una distribución normal
Para la columna GASTO_TOTAL las varianzas SÍ son homgéneas entre grupos, SI hay HOMOCEDASTICIDAD
H=8.0156, p=0.5326
Para la métrica GASTO_TOTAL, las medianas SI son iguales entre los grupos de ciudad (Kruskal-Wallis no significativo)


In [16]:
sa.decidir_test(clientes, 'ciudad', ['gasto_total'])

--- GASTO_TOTAL | normalidad=False | homocedastico=True | n_grupos=10 ---
H=8.0156, p=0.5326
Para la métrica GASTO_TOTAL, las medianas SI son iguales entre los grupos de ciudad (Kruskal-Wallis no significativo)


**Resultado:** H=8,0156; p=0,5326 → **no se rechaza H0**. Confirma que 
ciudad no diferencia el gasto real, tal como viste en Fase 3.

### Anexo: relaciones entre variables (también trazables a Fase 3)

In [18]:
sa.chi_cuadrado_independencia(df_pedidos, 'categoria_producto', 'devuelto')

chi2=2.639, p=0.6199, dof=4
NO hay evidencia de relación entre categoria_producto y devuelto


**Origen:** en Fase 3.4 la tasa de devolución apenas variaba por 
categoría (5,66%-6,06%). Resultado: χ²=2,6393; p=0,6199 → **sin 
relación**, confirmado.

In [19]:
sa.correlacion_regresion(clientes, 'ingresos_anuales', 'gasto_total')

Pearson r=-0.004 (p=0.7137)
Spearman rho=-0.004 (p=0.7423)
                            OLS Regression Results                            
Dep. Variable:            gasto_total   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.1346
Date:                Thu, 20 Aug 2026   Prob (F-statistic):              0.714
Time:                        23:47:18   Log-Likelihood:                -58381.
No. Observations:                7987   AIC:                         1.168e+05
Df Residuals:                    7985   BIC:                         1.168e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------

**Origen:** en la matriz de correlación de Fase 3.1, `ingresos_anuales` 
correlacionaba casi 0 con todo. Resultado: r=-0,0041 (p=0,7137), 
R²≈0,000017 → **sin relación lineal útil**, confirmado.

## Conclusiones — Fase 4

### Resumen de los tests realizados

| Caso | Pregunta | Nivel | Origen en Fase 3 | Test | p-valor | Resultado |
|---|---|---|---|---|---|---|
| A | gasto_total ~ segmento_cliente | Cliente | Gasto por segmento casi igual | Kruskal-Wallis | 0,4958 | Sin diferencia |
| B | importe_total_wz ~ canal | Pedido | Ticket medio casi igual por canal | Kruskal-Wallis | 0,4024 | Sin diferencia |
| C | importe_total_wz ~ envio_gratis | Pedido | Importe igual con/sin envío gratis | Mann-Whitney | 0,9573 | Sin diferencia |
| D.1 | margen ~ categoria_producto (producto, n=90) | Producto | Margen SÍ variaba por categoría | Kruskal-Wallis | 0,1145 | Sin diferencia (⚠️ poca potencia) |
| D.2 | margen ~ categoria_producto (pedido, n=52.000) | Pedido | Mismo hallazgo, más potencia | Kruskal-Wallis | 1,45e-222 | **Diferencia significativa** |
| E | gasto_total ~ ciudad | Cliente | Gasto por ciudad casi igual | Kruskal-Wallis | 0,5326 | Sin diferencia |
| Anexo | categoria_producto ~ devuelto | Pedido | Devolución casi igual por categoría | Chi-cuadrado | 0,6199 | Sin relación |
| Anexo | ingresos_anuales ~ gasto_total | Cliente | Correlación casi nula en matriz EDA | Correlación/regresión | 0,7137 | Sin relación |

### Interpretación general

De los 8 tests realizados, **7 confirman con rigor estadístico** lo que 
ya se había detectado visualmente en el EDA de Fase 3: segmento, canal, 
envío gratis, ciudad e ingresos anuales no explican el comportamiento 
de compra ni la devolución en este dataset.

El **Caso D es el único que revela una diferencia real** (margen entre 
categorías), y además ilustra un aprendizaje metodológico clave: el 
mismo patrón real puede parecer "no significativo" o "muy significativo" 
según la unidad de análisis y el tamaño de muestra disponible. Antes de 
concluir la ausencia de un efecto, es necesario comprobar que la muestra 
tiene suficiente potencia estadística para detectarlo.

### Conexión con el EDA (Fase 3)

Cada uno de los 8 casos de esta fase parte directamente de un hallazgo 
visual ya identificado en Fase 3, dando respaldo estadístico formal a 
esas observaciones. El caso del margen (D) refuerza además la 
recomendación de Fase 3 de investigar más a fondo la rentabilidad de 
electrónica frente al resto de categorías.